# Week 8 - Data Validation

This notebook focuses on **data validation from a data engineering perspective**.

At this point, the dataset has already passed through wrangling.
Now the goal is different:

> verify whether the cleaned dataset is really clean enough for database ingestion.


## Why validation

Wrangling tries to fix problems.
Validation checks whether those problems are actually resolved.

Even after cleaning, a dataset may still contain:
- unexpected nulls
- invalid categories
- values outside acceptable ranges
- schema drift
- rows that should not enter the database

In data engineering, validation is a **quality gate** before ingestion.


## Scope of this notebook

This is not an EDA notebook and not a machine learning notebook.

We will validate whether the data is structurally and operationally ready for loading.


In [55]:
# uncomment if you need any of those
#!pip install pandas pandera

## 1. Import libraries

In [56]:
from __future__ import annotations

from pathlib import Path
from typing import TypeAlias

import pandas as pd
import pandera.pandas as pa
from pandera import Check, Column, DataFrameSchema

## 2. Load the cleaned dataset from the previous wrangling lesson

We will reuse the cleaned Breast Cancer Wisconsin dataset.

In [57]:
DATA_PATH = Path('../../raw/breast-cancer-wisconsin/cleaned/breast-cancer-cleaned.csv') # adjust that to your actual cleaned dataset path
df_clean = pd.read_csv(DATA_PATH, sep=';')
df_clean.head()

,index,radius_1,texture_1,perimeter_1,area_1,smoothness_1,compactness_1,concavity_1,concave_points_1,symmetry_1,...,concavity_3,concave_points_3,symmetry_3,fractal_dimension_3,diagnosis,tumor_side,referral_type,hospital_region,source_file,source_dataset
0,0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,0.7119,0.2654,0.4601,0.11890,malignant,left,urgent,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,0.2416,0.1860,0.2750,0.08902,benign,right,routine,south,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
2,2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,0.4504,0.2430,0.3613,0.08758,malignant,left,urgent,east,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
3,3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,0.6869,0.2575,0.6638,0.17300,benign,right,routine,west,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
4,4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,0.4000,0.1625,0.2364,0.07678,malignant,left,routine,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)


## 3. Quick structural inspection

In [58]:
df_clean.shape

(569, 37)

In [59]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                569 non-null    int64  
 1   radius_1             569 non-null    float64
 2   texture_1            569 non-null    float64
 3   perimeter_1          569 non-null    float64
 4   area_1               569 non-null    float64
 5   smoothness_1         569 non-null    float64
 6   compactness_1        569 non-null    float64
 7   concavity_1          569 non-null    float64
 8   concave_points_1     569 non-null    float64
 9   symmetry_1           569 non-null    float64
 10  fractal_dimension_1  569 non-null    float64
 11  radius_2             569 non-null    float64
 12  texture_2            569 non-null    float64
 13  perimeter_2          569 non-null    float64
 14  area_2               569 non-null    float64
 15  smoothness_2         569 non-null    flo

In [60]:
df_clean.isna().sum()

index                  0
radius_1               0
texture_1              0
perimeter_1            0
area_1                 0
smoothness_1           0
compactness_1          0
concavity_1            0
concave_points_1       0
symmetry_1             0
fractal_dimension_1    0
radius_2               0
texture_2              0
perimeter_2            0
area_2                 0
smoothness_2           0
compactness_2          0
concavity_2            0
concave_points_2       0
symmetry_2             0
fractal_dimension_2    0
radius_3               0
texture_3              0
perimeter_3            0
area_3                 0
smoothness_3           0
compactness_3          0
concavity_3            0
concave_points_3       0
symmetry_3             0
fractal_dimension_3    0
diagnosis              0
tumor_side             0
referral_type          0
hospital_region        0
source_file            0
source_dataset         0
dtype: int64

## 4. Create a candidate ingestion dataset

To make validation visible, we will inject a few late-stage issues into a copy of the cleaned dataset.

This simulates what may happen after merges, transformations, or manual edits.

In [61]:
df_candidate = df_clean.copy()

df_candidate.loc[3, 'radius_1'] = -1
df_candidate.loc[7, 'diagnosis'] = 'borderline'
df_candidate.loc[11, 'tumor_side'] = None
df_candidate.loc[15, 'referral_type'] = 'emergency'
df_candidate.loc[20, 'hospital_region'] = 'central'
df_candidate.loc[25, 'area_3'] = -50
df_candidate.loc[31, 'smoothness_1'] = 2.5
df_candidate.loc[40, 'source_file'] = None

df_candidate.head(12)

,index,radius_1,texture_1,perimeter_1,area_1,smoothness_1,compactness_1,concavity_1,concave_points_1,symmetry_1,...,concavity_3,concave_points_3,symmetry_3,fractal_dimension_3,diagnosis,tumor_side,referral_type,hospital_region,source_file,source_dataset
0,0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,...,0.7119,0.26540,0.4601,0.11890,malignant,left,urgent,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,...,0.2416,0.18600,0.2750,0.08902,benign,right,routine,south,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
2,2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,...,0.4504,0.24300,0.3613,0.08758,malignant,left,urgent,east,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
3,3,-1.00,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,...,0.6869,0.25750,0.6638,0.17300,benign,right,routine,west,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
4,4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,...,0.4000,0.16250,0.2364,0.07678,malignant,left,routine,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
5,5,12.45,18.95,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,...,0.5355,0.17410,0.3985,0.12440,benign,right,urgent,south,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
6,6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,...,0.3784,0.19320,0.3063,0.08368,malignant,left,urgent,east,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
7,7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,...,0.2678,0.15560,0.3196,0.11510,borderline,left,routine,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
8,8,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,...,0.5390,0.20600,0.4378,0.10720,malignant,left,urgent,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
9,9,12.46,24.04,83.97,475.9,0.11860,0.23960,0.22730,0.08543,0.2030,...,1.1050,0.22100,0.4366,0.20750,malignant,left,urgent,south,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)


## 5. First layer: lightweight validation with type annotations

Before using a schema library, it is useful to define small explicit validation helpers.

This gives us readable Python functions and makes assumptions visible.

### 5.1 Define typed validation rules

In [62]:
RangeRule: TypeAlias = tuple[float | None, float | None]

required_columns: list[str] = [
    'index', 'radius_1', 'texture_1', 'perimeter_1', 'area_1', 'smoothness_1',
    'compactness_1', 'concavity_1', 'concave_points_1', 'symmetry_1',
    'fractal_dimension_1', 'radius_2', 'texture_2', 'perimeter_2', 'area_2',
    'smoothness_2', 'compactness_2', 'concavity_2', 'concave_points_2',
    'symmetry_2', 'fractal_dimension_2', 'radius_3', 'texture_3', 'perimeter_3',
    'area_3', 'smoothness_3', 'compactness_3', 'concavity_3', 'concave_points_3',
    'symmetry_3', 'fractal_dimension_3', 'diagnosis', 'tumor_side',
    'referral_type', 'hospital_region', 'source_file', 'source_dataset'
]

required_non_null: list[str] = [
    'radius_1', 'texture_1', 'area_1', 'smoothness_1', 'radius_3', 'area_3',
    'diagnosis', 'tumor_side', 'referral_type', 'hospital_region', 'source_file'
]

numeric_rules: dict[str, RangeRule] = {
    'radius_1': (0, 50),
    'texture_1': (0, 50),
    'area_1': (0, None),
    'smoothness_1': (0, 1),
    'radius_3': (0, 60),
    'area_3': (0, None),
    'smoothness_3': (0, 1),
}

allowed_diagnosis: set[str] = {'benign', 'malignant'}
allowed_tumor_side: set[str] = {'left', 'right'}
allowed_referral_type: set[str] = {'urgent', 'routine'}
allowed_hospital_region: set[str] = {'north', 'south', 'east', 'west'}

### 5.2 Create small validation functions

In [63]:
def find_missing_columns(df: pd.DataFrame, required: list[str]) -> list[str]:
    return [col for col in required if col not in df.columns]

def find_rows_with_required_nulls(df: pd.DataFrame, required: list[str]) -> pd.Series:
    return df[required].isna().any(axis=1)

def find_rows_outside_numeric_rules(df: pd.DataFrame, rules: dict[str, RangeRule]) -> pd.Series:
    invalid_mask = pd.Series(False, index=df.index)
    for col, (min_value, max_value) in rules.items():
        if min_value is not None:
            invalid_mask |= df[col] < min_value
        if max_value is not None:
            invalid_mask |= df[col] > max_value
    return invalid_mask

def find_rows_with_invalid_categories(df: pd.DataFrame) -> pd.Series:
    invalid_mask = pd.Series(False, index=df.index)
    invalid_mask |= ~df['diagnosis'].isin(allowed_diagnosis)
    invalid_mask |= ~df['tumor_side'].isin(allowed_tumor_side)
    invalid_mask |= ~df['referral_type'].isin(allowed_referral_type)
    invalid_mask |= ~df['hospital_region'].isin(allowed_hospital_region)
    return invalid_mask

### 5.3 Run the lightweight validation

In [64]:
missing_columns = find_missing_columns(df_candidate, required_columns)
null_rows = find_rows_with_required_nulls(df_candidate, required_non_null)
invalid_numeric_rows = find_rows_outside_numeric_rules(df_candidate, numeric_rules)
invalid_category_rows = find_rows_with_invalid_categories(df_candidate)

summary = {
    'missing_columns': missing_columns,
    'rows_with_required_nulls': int(null_rows.sum()),
    'rows_outside_numeric_rules': int(invalid_numeric_rows.sum()),
    'rows_with_invalid_categories': int(invalid_category_rows.sum()),
    'duplicated_rows': int(df_candidate.duplicated().sum()),
}

summary

{'missing_columns': [],
 'rows_with_required_nulls': 2,
 'rows_outside_numeric_rules': 4,
 'rows_with_invalid_categories': 4,
 'duplicated_rows': 0}

This first layer is already helpful, but it still depends on our own procedural logic.

For a more formal and reusable contract, we can define a schema.

## 6. Second layer: DataFrame validation with Pandera

Pandera allows us to express expected structure, types, and rules in a more declarative way.

This is a strong fit for ingestion pipelines that operate on DataFrames.

### Understanding the main Pandera concepts used here

Before reading the schema, it is useful to understand the meaning of some common Pandera rules:

- `Column(float, ...)` means that the column is expected to contain float values.
- `Check.in_range(a, b)` means that values must stay between `a` and `b`.
- `Check.ge(x)` means that values must be **greater than or equal to** `x`.
- `Check.isin([...])` means that values must belong to a fixed set of allowed categories.
- `nullable=False` means that the column should not contain missing values.
- `coerce=True` means that Pandera will try to coerce the column to the declared type before validating it.

So, in practice, the schema below is checking:
- whether required columns exist
- whether numeric fields are within acceptable operational ranges
- whether categorical fields use only allowed values
- whether critical fields are missing or not


### 6.1 Define a schema

In [65]:
cancer_schema = DataFrameSchema({
    'index': Column(int, Check.ge(0), nullable=False),
    'radius_1': Column(float, Check.in_range(0, 50), nullable=False),
    'texture_1': Column(float, Check.in_range(0, 50), nullable=False),
    'perimeter_1': Column(float, Check.ge(0), nullable=False),
    'area_1': Column(float, Check.ge(0), nullable=False),
    'smoothness_1': Column(float, Check.in_range(0, 1), nullable=False),
    'compactness_1': Column(float, Check.ge(0), nullable=False),
    'concavity_1': Column(float, Check.ge(0), nullable=False),
    'concave_points_1': Column(float, Check.ge(0), nullable=False),
    'symmetry_1': Column(float, Check.ge(0), nullable=False),
    'fractal_dimension_1': Column(float, Check.ge(0), nullable=False),
    'radius_2': Column(float, Check.ge(0), nullable=False),
    'texture_2': Column(float, Check.ge(0), nullable=False),
    'perimeter_2': Column(float, Check.ge(0), nullable=False),
    'area_2': Column(float, Check.ge(0), nullable=False),
    'smoothness_2': Column(float, Check.ge(0), nullable=False),
    'compactness_2': Column(float, Check.ge(0), nullable=False),
    'concavity_2': Column(float, Check.ge(0), nullable=False),
    'concave_points_2': Column(float, Check.ge(0), nullable=False),
    'symmetry_2': Column(float, Check.ge(0), nullable=False),
    'fractal_dimension_2': Column(float, Check.ge(0), nullable=False),
    'radius_3': Column(float, Check.in_range(0, 60), nullable=False),
    'texture_3': Column(float, Check.ge(0), nullable=False),
    'perimeter_3': Column(float, Check.ge(0), nullable=False),
    'area_3': Column(float, Check.ge(0), nullable=False),
    'smoothness_3': Column(float, Check.in_range(0, 1), nullable=False),
    'compactness_3': Column(float, Check.ge(0), nullable=False),
    'concavity_3': Column(float, Check.ge(0), nullable=False),
    'concave_points_3': Column(float, Check.ge(0), nullable=False),
    'symmetry_3': Column(float, Check.ge(0), nullable=False),
    'fractal_dimension_3': Column(float, Check.ge(0), nullable=False),
    'diagnosis': Column(str, Check.isin(['benign', 'malignant']), nullable=False),
    'tumor_side': Column(str, Check.isin(['left', 'right']), nullable=False),
    'referral_type': Column(str, Check.isin(['urgent', 'routine']), nullable=False),
    'hospital_region': Column(str, Check.isin(['north', 'south', 'east', 'west']), nullable=False),
    'source_file': Column(str, nullable=False),
    'source_dataset': Column(str, nullable=False),
}, coerce=True)

### 6.2 Validate the candidate dataset

In [66]:
try:
    cancer_schema.validate(df_candidate)
    print('Validation passed.')
except pa.errors.SchemaError as exc:
    print(type(exc).__name__)
    print(exc)

SchemaError
Column 'radius_1' failed element-wise validator number 0: in_range(0, 50) failure cases: -1.0, 99.9


The first schema error is useful, but in real pipelines we usually want a fuller diagnostic report.

By default, Pandera stops validation as soon as it finds the **first error**.
This behavior is often called **fail fast**.

That can be useful for quick debugging, but it is not always ideal in ingestion pipelines because one bad record may hide many other problems in the dataset.

When we use `lazy=True`, Pandera does **not stop at the first failure**.
Instead, it continues checking the DataFrame and collects multiple validation problems before raising the final exception.

In practice, `lazy=True` is helpful when we want:
- a broader diagnostic report
- a list of several failing rows or columns at once
- better visibility into the real quality state of the dataset


### 6.3 Collect multiple failures at once

In [67]:
try:
    cancer_schema.validate(df_candidate, lazy=True)
except pa.errors.SchemaErrors as exc:
    pandera_failures = exc.failure_cases.copy()
    pandera_failures.head(20)

## 7. Build ingestion outputs

A simple engineering pattern is:
- rows that pass validation go forward
- rows that fail validation are separated for inspection

### 7.1 Create a combined row-level failure mask

In [68]:
combined_invalid_mask = (
    null_rows
    | invalid_numeric_rows
    | invalid_category_rows
    | df_candidate.duplicated()
)

df_ready = df_candidate.loc[~combined_invalid_mask].copy()
df_rejected = df_candidate.loc[combined_invalid_mask].copy()

df_ready.shape, df_rejected.shape

((560, 37), (9, 37))

### 7.2 Inspect the accepted records

In [69]:
df_ready.head()

,index,radius_1,texture_1,perimeter_1,area_1,smoothness_1,compactness_1,concavity_1,concave_points_1,symmetry_1,...,concavity_3,concave_points_3,symmetry_3,fractal_dimension_3,diagnosis,tumor_side,referral_type,hospital_region,source_file,source_dataset
0,0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,0.7119,0.2654,0.4601,0.11890,malignant,left,urgent,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,0.2416,0.1860,0.2750,0.08902,benign,right,routine,south,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
2,2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,0.4504,0.2430,0.3613,0.08758,malignant,left,urgent,east,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
4,4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,0.4000,0.1625,0.2364,0.07678,malignant,left,routine,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
5,5,12.45,18.95,82.57,477.1,0.12780,0.17000,0.1578,0.08089,0.2087,...,0.5355,0.1741,0.3985,0.12440,benign,right,urgent,south,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)


### 7.3 Inspect the rejected records

In [70]:
df_rejected.head()

,index,radius_1,texture_1,perimeter_1,area_1,smoothness_1,compactness_1,concavity_1,concave_points_1,symmetry_1,...,concavity_3,concave_points_3,symmetry_3,fractal_dimension_3,diagnosis,tumor_side,referral_type,hospital_region,source_file,source_dataset
3,3,-1.00,20.38,77.58,386.1,0.142500,0.2839,0.24140,0.10520,0.2597,...,0.6869,0.25750,0.6638,0.17300,benign,right,routine,west,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
7,7,13.71,20.83,90.20,577.9,0.118900,0.1645,0.09366,0.05985,0.2196,...,0.2678,0.15560,0.3196,0.11510,borderline,left,routine,north,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
11,11,15.78,17.89,103.60,781.0,0.097100,0.1292,0.09954,0.06606,0.1842,...,0.3965,0.18100,0.3792,0.10480,malignant,None,routine,west,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
15,15,14.54,27.54,96.73,658.8,0.113900,0.1595,0.16390,0.07364,0.2303,...,0.7026,0.17120,0.4218,0.13410,malignant,right,emergency,west,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)
20,20,13.08,15.71,85.63,520.0,0.095785,0.1270,0.04568,0.03110,0.1967,...,0.1890,0.07283,0.3184,0.08183,benign,left,routine,central,breast-cancer-dirty.csv,Breast Cancer Wisconsin (dirty classroom version)


### 7.4 Export the outputs

In [71]:
output_dir = Path('../../raw/breast-cancer-wisconsin/validated')
output_dir.mkdir(parents=True, exist_ok=True)

ready_path = output_dir / 'breast-cancer-ready-for-ingestion.csv'
rejected_path = output_dir / 'breast-cancer-rejected-records.csv'

df_ready.to_csv(ready_path, index=False)
df_rejected.to_csv(rejected_path, index=False)

ready_path, rejected_path

(WindowsPath('../../raw/breast-cancer-wisconsin/validated/breast-cancer-ready-for-ingestion.csv'),
 WindowsPath('../../raw/breast-cancer-wisconsin/validated/breast-cancer-rejected-records.csv'))

## 8. Why this improves ingestion pipelines

Validation improves ingestion pipelines because it:
- prevents clearly bad records from reaching the database
- makes data assumptions explicit
- creates a repeatable quality gate
- reduces silent failures downstream
- makes rejection traceable and auditable

In practice, validation helps transform a cleaned dataset into an ingestion-ready dataset with more confidence.


## 9. A note about type annotations

Type annotations are useful because they make code easier to read and maintain.
They communicate developer intent and help document what a function expects.

However, **Python does not enforce type annotations at runtime by default**.
This means that a function annotated with `list[str]` or `dict[str, RangeRule]` can still receive the wrong kinds of values unless we validate them explicitly.

That is why type annotations alone are not enough for ingestion pipelines.
They help us write clearer code, but tools such as **Pandera** are still needed to validate the actual data.

During development, it is a good idea to use a static type checker in the editor.
In VS Code, **Pylance** can highlight many type inconsistencies before the code runs.


`IMPORTANT`: A more production-ready workflow would strengthen this notebook with clearer logging, automated tests, data contracts, and more explicit rejection reporting. Another common pattern is to connect Pandera validation directly to an orchestration step in the ingestion pipeline.

References:
- [Pandera documentation](https://pandera.readthedocs.io/en/stable/)
- [Python typing documentation](https://docs.python.org/3/library/typing.html)
- [Pandas documentation](https://pandas.pydata.org/docs/)
- [Pylance documentation](https://marketplace.visualstudio.com/items?itemName=ms-python.vscode-pylance)

## Final reflection

The key idea is:

> wrangling prepares the data, but validation proves whether the data is really ready.

For this class, a good practical combination is:
- simple typed helper functions for quick checks
- Pandera for schema-based validation
- a final split between accepted and rejected rows for the pipeline
